# EPINUC PTM colocalization — TIF image runs

Runs the **existing** `epinuc_colocalization` pipeline over the EpiVision TIF collections in
`/Volumes/scBC/EpiVision/Images/{NUC388,NUC389,NUC405}` — no analysis logic is changed, only the
file-loading layer (`epinuc_tiff_loader`).

**Data model** (per the *PTM Image Analysis Requirements* doc). Each flowcell lane `chN` is one
EPINUC channel = a nucleosome template + a red/blue antibody pair:

| pipeline role | files | cycles |
|---|---|---|
| `nucleosome` (green, reference) | base `N`, laser `G` | cycle 1 only → reused as static template |
| `R_PTM` (red)  | base `Z1/Z3/Z5`, laser `R` | 1–5 |
| `B_PTM` (blue) | base `Z2/Z4/Z6`, laser `B` | 1–5 |

The loader maps **position → FOV/scene, cycle → time point, role → channel**, so the pipeline's
native cumulative new-only counting runs across the 5 antibody cycles. The `img0..img10` index in
the filename is redundant (it just enumerates the 11 images per position) and is ignored — slots
are keyed on `(position, role, cycle)`. Retry/attempt duplicates for a slot are max-projected.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import epinuc_colocalization as ep
import epinuc_tiff_loader as tl

# Optional: load tuned spot-detection / colocalization parameters if present.
if os.path.exists('epinuc_config.json'):
    ep.apply_config(ep.load_config('epinuc_config.json'))

# Point the pipeline's channel map at the loader's identity roles, disable the across-cycle
# coarse bead-vote (the shared fixed nucleosome template already defines one common frame — see
# configure_pipeline docstring), and enable blob/streak artifact masking. Pass pixel_size_um=...
# if the instrument value is known, or artifact_masking=False to disable.
tl.configure_pipeline()

IMAGES_ROOT = '/Volumes/scBC/EpiVision/Images'
RUN_INFO_XLSX = '/Volumes/scBC/EpiVision/Images/Runinformation.xlsx'
OUTPUT_ROOT = 'per_run_output'

# QUICK=True: a fast smoke run (first run, first lane, 2 FOVs). Set False for the full analysis.
QUICK = True
ep.SAVE_QC_FIGURES = False   # set True to also emit QC PNGs (slow)
print('pipeline roles:', ep.CHANNEL_MAP, '| TIME_COARSE_ALIGN =', ep.TIME_COARSE_ALIGN,
      '| ARTIFACT_MASKING =', ep.ARTIFACT_MASKING)

## 1. Discover runs & lanes, sanity-check the parsed inventory

In [ ]:
import time

# The /Volumes/scBC SMB share drops out intermittently. Confirm it's mounted (retry a few
# times) before discovery, so a transient blip gives a clear message, not an empty run list.
for _ in range(5):
    if os.path.isdir(IMAGES_ROOT):
        break
    time.sleep(1)
if not os.path.isdir(IMAGES_ROOT):
    raise RuntimeError(f"{IMAGES_ROOT} is not reachable — the /Volumes/scBC network share "
                       f"appears unmounted. Remount it and re-run this cell.")

run_dirs = sorted(d for d in glob.glob(os.path.join(IMAGES_ROOT, '*')) if os.path.isdir(d))
run_index = {os.path.basename(d): tl.index_run(d)
             for d in tqdm(run_dirs, desc='indexing runs')}
if not run_index:
    raise RuntimeError(f"No run folders found under {IMAGES_ROOT} (share may have dropped mid-listing).")

rows = []
for run, idx in run_index.items():
    for lane in tl.lanes_in_run(idx):
        li = idx[idx['lane'] == lane]
        rows.append({
            'run': run, 'lane': lane,
            'sample': sorted(li['sample'].unique()),
            'n_pos': li['pos'].nunique(),
            'red_base': sorted(li[li.role=='R_PTM']['base'].unique()),
            'blue_base': sorted(li[li.role=='B_PTM']['base'].unique()),
            'nuc_cycles': sorted(li[li.role=='nucleosome']['cycle'].unique()),
            'ab_cycles': sorted(li[li.role!='nucleosome']['cycle'].unique()),
            'n_files': len(li),
        })
inventory = pd.DataFrame(rows)
inventory

## 2. Run → antibody labels from `Run information.xlsx` (best effort)

The `/Volumes/scBC` mount is flaky, so this reads the sheet defensively and falls back to
`NUC###_chN` labels if it can't. **Inspect the printed sheet**, then fill `RUN_LANE_ANTIBODY`
below (keyed by `(run, lane)`) with the antibody/sample names so they appear in `sample_id`.

In [ ]:
pd.read_excel(RUN_INFO_XLSX)

In [ ]:
tl.load_run_labels(RUN_INFO_XLSX)

In [ ]:
run_info = tl.load_run_labels(RUN_INFO_XLSX)
if run_info is not None:
    display(run_info)
else:
    print('Run information.xlsx unavailable — using NUC###_chN labels.')

# Fill after inspecting the sheet, e.g. {('NUC388','ch1'): 'H3K27me3/H3K9ac'}. Missing keys
# fall back to NUC###_chN. Lane->dye-slot reference: ch1/ch4->Z1(R)/Z2(B), ch2/ch5->Z3/Z4,
# ch3/ch6->Z5/Z6; ch1-3=sample C001, ch4-6=C002.
RUN_LANE_ANTIBODY = {}

def label_for(run, lane):
    return tl.lane_label(run, lane, RUN_LANE_ANTIBODY.get((run, lane)))

## 3. Quick validation — one lane, a couple of FOVs

Confirm sane per-image spot counts (nucleosome ~thousands, R/B ~hundreds) and that the
**cumulative nucleosome count stays stable** across cycles (the template is fixed) while R/B
accumulate.

In [ ]:
val_run = list(run_index)[0]
val_lane = tl.lanes_in_run(run_index[val_run])[0]
acc = tl.run_channel(run_index[val_run], val_lane, sample_id=label_for(val_run, val_lane),
                     scenes=range(1), show_progress=True)

counts = pd.DataFrame(acc.counts)
show = [c for c in ['scene','time','n_nucleosome','n_R','n_B',
                    'n_R_nucleosome','n_B_nucleosome','n_R_B_colocalized','n_R_B_nucleosome_triple']
        if c in counts.columns]
print('per-image counts:'); display(counts[show])

cum = pd.DataFrame(acc.cumulative)
cum_show = ['scene','time','cumulative_n_nucleosomes','cumulative_n_R_PTMs','cumulative_n_B_PTMs',
            'cumulative_n_R_B_colocalized','cumulative_n_triple_colocalized']
print('cumulative across cycles (FOV 0):'); display(cum[cum.scene==0][cum_show])

## Diagnostics — inspect one FOV (images & overlays)

Visual QA on a single field of view: raw channels + RGB overlay, bead detection, spot detection
with a threshold sweep, and the colocalization overlay. Change `DIAG_RUN/DIAG_LANE/DIAG_CYCLE/DIAG_FOV`
in the first cell to inspect any FOV. For live slider-driven tuning, use `streamlit run epinuc_tiff_gui.py`.

In [ ]:
import matplotlib.pyplot as plt

ROLES = ['nucleosome', 'R_PTM', 'B_PTM']
ROLE_COLOR = {'nucleosome': 'lime', 'R_PTM': 'red', 'B_PTM': 'deepskyblue'}

# Pick a FOV to inspect (defaults to the validation lane / cycle 1 / FOV 0).
DIAG_RUN, DIAG_LANE, DIAG_CYCLE, DIAG_FOV = val_run, val_lane, 1, 0

planes = tl.fov_planes(run_index[DIAG_RUN], DIAG_LANE, DIAG_CYCLE, DIAG_FOV)
green, red, blue = planes['nucleosome'], planes['R_PTM'], planes['B_PTM']

fig, axes = plt.subplots(1, 4, figsize=(18, 4.6))
for ax, r, cm in zip(axes[:3], ROLES, ['Greens', 'Reds', 'Blues']):
    im = planes[r]
    if im is not None:
        ax.imshow(ep._norm01(im), cmap=cm)
        ax.set_title(f'{r}\nmed={np.median(im):.0f}  max={im.max():.0f}')
    else:
        ax.set_title(f'{r}: missing')
    ax.axis('off')
axes[3].imshow(ep.rgb_overlay(green, red, blue)); axes[3].set_title('RGB overlay'); axes[3].axis('off')
fig.suptitle(f'{DIAG_RUN} {DIAG_LANE} cycle{DIAG_CYCLE} FOV{DIAG_FOV}', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Bead detection per channel + the cross-channel-confirmed fiducials used for
# registration / bead-exclusion. A flooded detector (thousands) means no clear fiducials.
beads = {r: ep.detect_beads(planes[r], '', DIAG_FOV, 0, r) for r in ROLES}
conf = ep.confirmed_bead_coords([beads['nucleosome'], beads['R_PTM'], beads['B_PTM']])
print('raw beads per channel:', {r: len(beads[r]) for r in ROLES}, '| cross-channel confirmed:', len(conf))

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
axes[0].imshow(ep._norm01(green), cmap='gray')
b = beads['nucleosome']
if len(b):
    axes[0].scatter(b['x'], b['y'], s=45, marker='o', facecolors='none', edgecolors='orange', linewidths=1)
axes[0].set_title(f'green + per-channel beads ({len(b)})'); axes[0].axis('off')
axes[1].imshow(ep._norm01(green), cmap='gray')
if len(conf):
    axes[1].scatter(conf[:, 1], conf[:, 0], s=70, marker='o', facecolors='none', edgecolors='yellow', linewidths=1.3)
axes[1].set_title(f'green + {len(conf)} CONFIRMED beads'); axes[1].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Blob/streak artifact mask. Large bright regions (debris/aggregates/reflections/streaks) are
# bright in every channel, get used as false fiducials and seed false spots/triples, so the
# pipeline drops beads & spots inside them and blanks the region for background. The AREA is the
# discriminator (molecules ~1-9px, beads ~10-50px, blobs hundreds-thousands). Tune via
# ep.ARTIFACT_BRIGHT_PCT / ARTIFACT_MIN_AREA_PX / ARTIFACT_DILATION_PX.
#
# This is the ALL-CYCLES mask the pipeline actually applies: debris is stuck on the flowcell, the
# green template exists only at cycle 1, and a channel-specific artifact can fade in a later cycle
# — so one mask is unioned over the template + every cycle's R/B and reused at every timepoint.
from skimage.measure import label as _label
from matplotlib.colors import ListedColormap

art = tl.lane_artifact_mask(run_index[DIAG_RUN], DIAG_LANE, DIAG_FOV)
if art is None or not art.any():
    print('No blob/streak artifacts detected in this FOV.')
else:
    conf_in = int(art[conf[:, 0].astype(int), conf[:, 1].astype(int)].sum()) if len(conf) else 0
    beads_in = sum(int(ep.exclude_points_in_mask(beads[r], art)[1]) for r in ROLES)
    print(f'all-cycles artifact mask covers {100*art.mean():.3f}% of the FOV in {_label(art).max()} '
          f'region(s); drops {conf_in}/{len(conf)} confirmed fiducials and {beads_in} raw beads '
          f'that sat on blobs')

    overlay = np.ma.masked_where(~art, np.ones_like(green))
    fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
    axes[0].imshow(ep._norm01(green), cmap='gray')
    axes[0].imshow(overlay, cmap=ListedColormap(['red']), alpha=0.45)
    axes[0].set_title('green + artifact mask (red)'); axes[0].axis('off')
    axes[1].imshow(ep.rgb_overlay(green, red, blue))
    axes[1].contour(art, levels=[0.5], colors='white', linewidths=1.0)
    axes[1].set_title('RGB overlay + mask outline'); axes[1].axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
# Spot detection — mirrors _process_one_fov exactly, so what you see is what the pipeline counts:
#   1. drop artifact beads BEFORE cross-channel confirmation (no false fiducials)
#   2. register R/B onto the green template
#   3. detect spots with the artifact region blanked from the background estimate
#   4. drop spots near beads, then drop spots inside the artifact mask
beads_m = {r: ep.exclude_points_in_mask(beads[r], art)[0] for r in ROLES} if art is not None else dict(beads)
conf_m = ep.confirmed_bead_coords([beads_m['nucleosome'], beads_m['R_PTM'], beads_m['B_PTM']])

tf_id = ep.ChannelTransform(method='reference', success=True)
tf = {'nucleosome': tf_id}
tf['R_PTM'] = (ep.estimate_channel_transform(beads_m['nucleosome'], beads_m['R_PTM'], green, red, ep.REGISTRATION_MODE)
               if red is not None else tf_id)
tf['B_PTM'] = (ep.estimate_channel_transform(beads_m['nucleosome'], beads_m['B_PTM'], green, blue, ep.REGISTRATION_MODE)
               if blue is not None else tf_id)

spots = {r: ep.detect_spots(planes[r], '', DIAG_FOV, 0, r, tf[r], bead_yx=conf_m, artifact_mask=art)
         for r in ROLES}
raw_counts = {r: len(spots[r]) for r in ROLES}
if ep.EXCLUDE_BEADS_FROM_SPOTS and len(conf_m):
    for r in ROLES:
        spots[r], _ = ep.exclude_spots_near_beads(spots[r], conf_m)
if art is not None and art.any():
    for r in ROLES:
        spots[r], _ = ep.exclude_points_in_mask(spots[r], art)
print('spots before masking:', raw_counts)
print('spots after bead + artifact exclusion:', {r: len(spots[r]) for r in ROLES})

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, r in zip(axes, ROLES):
    if planes[r] is not None:
        ax.imshow(ep._norm01(planes[r]), cmap='gray')
    if art is not None and art.any():
        ax.contour(art, levels=[0.5], colors='yellow', linewidths=0.7)
    d = spots[r]
    if len(d):
        ax.scatter(d['x'], d['y'], s=8, marker='+', c=ROLE_COLOR[r], linewidths=0.6)
    ax.set_title(f'{r}: {len(d)} kept of {raw_counts[r]} (k={ep.SPOT_DETECTION_SNR[r]})'); ax.axis('off')
fig.suptitle('spots kept after masking (yellow = artifact regions, now empty)', y=1.01)
plt.tight_layout(); plt.show()

# Threshold sweep — also artifact-masked, so the curve reflects real (kept) spots.
fig, ax = plt.subplots(figsize=(7, 4))
ks = np.arange(4, 20.1, 1.0)
for r in ROLES:
    saved = ep.SPOT_DETECTION_SNR[r]; cnts = []
    for k in ks:
        ep.SPOT_DETECTION_SNR[r] = float(k)
        s = ep.detect_spots(planes[r], '', DIAG_FOV, 0, r, tf_id, bead_yx=conf_m, artifact_mask=art)
        if art is not None and art.any():
            s, _ = ep.exclude_points_in_mask(s, art)
        cnts.append(len(s))
    ep.SPOT_DETECTION_SNR[r] = saved
    ax.plot(ks, cnts, '-o', ms=3, label=r, color=ROLE_COLOR[r]); ax.axvline(saved, color=ROLE_COLOR[r], ls=':', alpha=0.4)
ax.set_yscale('log'); ax.set_xlabel('k (σ above noise)'); ax.set_ylabel('spot count (artifact-masked)')
ax.set_title('Spot count vs detection threshold k (dotted = current)'); ax.legend(); plt.show()

In [ ]:
# Colocalization overlay on the nucleosome (green) image: faint dots = registered spots,
# rings = colocalization events at the nucleosome position (R–N, B–N, and triples).
events, cc = ep.colocalize(spots['nucleosome'], spots['R_PTM'], spots['B_PTM'], '', DIAG_FOV, 0)
print('coloc counts:', {k: cc[k] for k in cc if k.startswith('n_')})

fig, ax = plt.subplots(figsize=(8, 8))
if green is not None:
    ax.imshow(ep._norm01(green), cmap='gray')
for r in ROLES:
    d = spots[r]
    if len(d):
        ax.scatter(d['registered_x'], d['registered_y'], s=8, c=ROLE_COLOR[r], alpha=0.35, label=r)
for etype, lab, color, size in [('R_nucleosome','R–N','orange',70),
                                ('B_nucleosome','B–N','magenta',130),
                                ('R_B_nucleosome_triple','triple','yellow',230)]:
    e = events[events['event_type'] == etype] if len(events) else events
    if len(e):
        ax.scatter(e['nucleosome_x'], e['nucleosome_y'], s=size, facecolors='none',
                   edgecolors=color, linewidths=1.6, label=f'{lab} ({len(e)})')
ax.legend(fontsize=8, loc='upper right', framealpha=0.6)
ax.set_title(f'Colocalization (registered) — {DIAG_RUN} {DIAG_LANE} cyc{DIAG_CYCLE} FOV{DIAG_FOV}')
ax.axis('off'); plt.show()

### Per-cycle & cumulative trends

In [ ]:
# Per-cycle trends for the validation lane (FOV 0): raw per-image counts and the pipeline's
# cumulative new-only counts. Nucleosome should be flat (fixed template); R/B accumulate.
c = pd.DataFrame(acc.counts); u = pd.DataFrame(acc.cumulative)
c0 = c[c.scene == 0].sort_values('time'); u0 = u[u.scene == 0].sort_values('time')

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.5))
for col, lab, color in [('n_nucleosomes','N (nucleosome)','green'),('n_R_PTMs','R','red'),('n_B_PTMs','B','blue')]:
    if col in c0: axL.plot(c0['time']+1, c0[col], '-o', label=lab, color=color)
axL.set_xlabel('cycle'); axL.set_ylabel('per-image spot count')
axL.set_title(f'{val_run} {val_lane} FOV0 — raw spots per cycle'); axL.legend(); axL.set_xticks(range(1, len(c0)+1))

for col, lab, color in [('cumulative_n_nucleosomes','N','green'),('cumulative_n_R_PTMs','R','red'),
                        ('cumulative_n_B_PTMs','B','blue'),('cumulative_n_R_B_colocalized','R∩B','purple'),
                        ('cumulative_n_triple_colocalized','triple','black')]:
    if col in u0: axR.plot(u0['time']+1, u0[col], '-o', label=lab, color=color)
axR.set_xlabel('cycle'); axR.set_ylabel('cumulative unique detections')
axR.set_title('Cumulative new-only across cycles'); axR.legend(); axR.set_xticks(range(1, len(u0)+1))
plt.tight_layout(); plt.show()

## 4. Full run — every run × every lane

Each lane's CSVs go to `per_run_output/<run>/<lane>/` (`per_image_counts.csv`,
`cumulative_new_counts.csv`, `colocalization_events.csv`, `sample_<id>_summary.csv`, …).
Set `QUICK = False` in the first cell for the complete set.

**Parallelism:** FOVs fan out across `ncores-1` workers (`ep.N_JOBS = -2`), so a single sample
still uses every core. Each FOV's whole 5-cycle series stays inside one worker, so the cumulative
new-only counting is unaffected — parallel results are bit-identical to serial.

In [ ]:
QUICK=False
scenes = range(2) if QUICK else None

jobs = ([(list(run_index)[0], tl.lanes_in_run(run_index[list(run_index)[0]])[0])] if QUICK
        else [(run, lane) for run, idx in run_index.items() for lane in tl.lanes_in_run(idx)])

In [ ]:
scenes

In [ ]:
# FOV-level multiprocessing. Lanes are handled one at a time (bounding memory), but each lane's
# FOVs fan out across ncores-1 worker processes (joblib loky) — so even a SINGLE sample / single
# lane saturates the machine. Each FOV's whole 5-cycle series stays in one worker, so the
# cumulative new-only counting is untouched; results are merged and exported per lane to
# per_run_output/<run>/<lane>/. n_jobs=ep.N_JOBS (-2 = all cores but one; -1 = all; 1 = serial).
print(f'workers: {tl.resolve_n_jobs(ep.N_JOBS)} of {os.cpu_count()} cores')
all_summaries = tl.run_channels_parallel(
    run_index, jobs, scenes=scenes, output_root=OUTPUT_ROOT,
    n_jobs=ep.N_JOBS, save_qc=ep.SAVE_QC_FIGURES, label_fn=label_for)
all_summaries.to_csv(os.path.join(OUTPUT_ROOT, 'all_lanes_summary.csv'), index=False)
all_summaries

In [ ]:
jobs

## 5. Results table in the EpiVision "Results Format" shape

One row per lane: `#N`, `#Ab1`(=R), `#Ab2`(=B), `#(Ab1&Ab2)`(=R↔B), `#(Ab1&Ab2&N)`(=triple),
`#(Ab1&N)`(=R↔N), `#(Ab2&N)`(=B↔N).

In [ ]:
results = all_summaries.rename(columns={
    'sample': 'sample_id',
    'cumulative_n_nucleosomes': '#N',
    'cumulative_n_R_PTMs': '#Ab1(R)',
    'cumulative_n_B_PTMs': '#Ab2(B)',
    'cumulative_n_R_B_colocalized': '#(Ab1&Ab2)',
    'cumulative_n_triple_colocalized': '#(Ab1&Ab2&N)',
    'cumulative_n_R_colocalized_with_nucleosome': '#(Ab1&N)',
    'cumulative_n_B_colocalized_with_nucleosome': '#(Ab2&N)',
})[['run','lane','sample_id','n_FOVs','n_timepoints',
    '#N','#Ab1(R)','#Ab2(B)','#(Ab1&Ab2)','#(Ab1&Ab2&N)','#(Ab1&N)','#(Ab2&N)']]
results.to_csv(os.path.join(OUTPUT_ROOT, 'results_format.csv'), index=False)
results